In [1]:
import sys

project_root = 'c:/big20/git/big20-ML-project2-team3/CreditCardFraud'

# sys.path에 추가 (모듈 import용)
if project_root not in sys.path:
    sys.path.insert(0, project_root)

In [2]:
import pandas as pd
import numpy  as np
from scipy.special import logit
import time
import warnings
warnings.filterwarnings("ignore")
from pathlib import Path

from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score, roc_curve, classification_report, confusion_matrix, recall_score, precision_score
from sklearn.ensemble import RandomForestClassifier, StackingClassifier
from hyperopt import fmin, tpe, hp, STATUS_OK, Trials

import matplotlib.pyplot as plt
plt.rcParams['font.family'] ='Malgun Gothic'
import seaborn as sns

import importlib
from utils import preprocessing
importlib.reload(preprocessing) # 모듈 reload
# importlib.reload(user_utils)

import utils.preprocessing as pp
import utils.user_utils    as uu
import utils.model_utils   as mu
import utils.evaluation    as ev
import utils.modeling      as mo
import utils.data_sampling as ds

In [3]:
# data loading
df = pp.ccf_load_data()

데이터 로드 성공: (284807, 31)


In [5]:
print(df.shape)
print(df.info())
print(df['Class'].value_counts())

(284807, 31)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 284807 entries, 0 to 284806
Data columns (total 31 columns):
 #   Column  Non-Null Count   Dtype  
---  ------  --------------   -----  
 0   Time    284807 non-null  float64
 1   V1      284807 non-null  float64
 2   V2      284807 non-null  float64
 3   V3      284807 non-null  float64
 4   V4      284807 non-null  float64
 5   V5      284807 non-null  float64
 6   V6      284807 non-null  float64
 7   V7      284807 non-null  float64
 8   V8      284807 non-null  float64
 9   V9      284807 non-null  float64
 10  V10     284807 non-null  float64
 11  V11     284807 non-null  float64
 12  V12     284807 non-null  float64
 13  V13     284807 non-null  float64
 14  V14     284807 non-null  float64
 15  V15     284807 non-null  float64
 16  V16     284807 non-null  float64
 17  V17     284807 non-null  float64
 18  V18     284807 non-null  float64
 19  V19     284807 non-null  float64
 20  V20     284807 non-null  float64
 2

In [4]:
df['Amount'].describe()

count    284807.000000
mean         88.349619
std         250.120109
min           0.000000
25%           5.600000
50%          22.000000
75%          77.165000
max       25691.160000
Name: Amount, dtype: float64

In [ ]:
import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, roc_auc_score, recall_score

# -----------------------------
# 1. Time 컬럼 제거
# -----------------------------
df_processed = df.drop(['Time'], axis=1)

# -----------------------------
# 2. Amount 로그 변환 + StandardScaler 적용
# -----------------------------
df_processed['Amount_log'] = np.log1p(df_processed['Amount'])  # log(Amount+1)
scaler = StandardScaler()
df_processed['Amount_scaled'] = scaler.fit_transform(df_processed['Amount_log'].values.reshape(-1, 1))

# 원본 Amount 제거
df_processed = df_processed.drop(['Amount', 'Amount_log'], axis=1)

# -----------------------------
# 3. 특성과 타겟 분리
# -----------------------------
X = df_processed.drop('Class', axis=1)
y = df_processed['Class']

# -----------------------------
# 4. 학습/테스트 분할
# -----------------------------
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=23
)

# -----------------------------
# 5. Logistic Regression 학습
# -----------------------------
log_reg = LogisticRegression(max_iter=1000, random_state=23)
log_reg.fit(X_train, y_train)

# -----------------------------
# 6. 예측 및 평가
# -----------------------------
y_pred = log_reg.predict(X_test)
y_pred_proba = log_reg.predict_proba(X_test)[:, 1]

print("\n[Classification Report]")
print(classification_report(y_test, y_pred))

print("ROC-AUC Score:", roc_auc_score(y_test, y_pred_proba))
print("Recall:", recall_score(y_test, y_pred))

result = '''
[Classification Report]
              precision    recall  f1-score   support

           0       1.00      1.00      1.00     56864
           1       0.88      0.65      0.75        98

    accuracy                           1.00     56962
   macro avg       0.94      0.83      0.87     56962
weighted avg       1.00      1.00      1.00     56962

ROC-AUC Score: 0.9590699039886073
Recall: 0.6530612244897959
'''


[Classification Report]
              precision    recall  f1-score   support

           0       1.00      1.00      1.00     56864
           1       0.88      0.65      0.75        98

    accuracy                           1.00     56962
   macro avg       0.94      0.83      0.87     56962
weighted avg       1.00      1.00      1.00     56962

ROC-AUC Score: 0.9590699039886073
Recall: 0.6530612244897959


In [ ]:
import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression

def preprocess_and_train_logreg(df, model_name="LogisticRegression"):
    """
    df: 원본 데이터셋
    model_name: 모델 이름 (기본값 LogisticRegression)
    """
    # -----------------------------
    # 1. Time 컬럼 제거
    # -----------------------------
    df_processed = df.drop(['Time'], axis=1)

    # -----------------------------
    # 2. Amount 로그 변환 + StandardScaler 적용
    # -----------------------------
    df_processed['Amount_log'] = np.log1p(df_processed['Amount'])  # log(Amount+1)
    scaler = StandardScaler()
    df_processed['Amount_scaled'] = scaler.fit_transform(
        df_processed['Amount_log'].values.reshape(-1, 1)
    )

    # 원본 Amount, Amount_log 제거
    df_processed = df_processed.drop(['Amount', 'Amount_log'], axis=1)

    # -----------------------------
    # 3. 특성과 타겟 분리
    # -----------------------------
    X = df_processed.drop('Class', axis=1)
    y = df_processed['Class']

    # -----------------------------
    # 4. 학습/테스트 분할
    # -----------------------------
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, stratify=y, random_state=23
    )

    # -----------------------------
    # 5. Logistic Regression 학습
    # -----------------------------
    log_reg = LogisticRegression(max_iter=1000, random_state=23)
    log_reg.fit(X_train, y_train)

    # -----------------------------
    # 6. 평가 (사용자 정의 함수 호출)
    # -----------------------------
    uu.get_model_train_eval(log_reg, model_name, X_train, X_test, y_train, y_test)

    return log_reg, X_train, X_test, y_train, y_test

### 데이터 로드
df = pp.ccf_load_data()

### 함수 실행
log_reg, X_train, X_test, y_train, y_test = preprocess_and_train_logreg(df, model_name="LogReg_with_LogAmount")